# WorldsIndex reproducibility bundle: HD 209458 b transit ephemeris

This notebook uses only the files in this folder. It re-applies the published selection
policy to the planet's catalog rows, recomputes the predicted mid-transit times with their
1-sigma uncertainty, compares them with the published ephemeris row and draws the
uncertainty growth over one year.

In [ ]:
import json, math
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

detail = json.loads(Path("records/HD-209458-b.json").read_text())
eph = json.loads(Path("records/ephemeris-row.json").read_text())
config = json.loads(Path("analysis/config.json").read_text())
row = dict(zip(eph["columns"], eph["row"]))
print(row["name"], row["sourceId"], row["periodDays"], row["periodErrDays"], row["t0Bjd"], row["t0ErrDays"])

In [ ]:
# Selection policy, step by step (analysis/config.json quotes it): PSCompPars if it reports
# both a period and a mid-transit time; else the NASA PS default set; else ... else the TOI,
# K2 or KOI row with the smallest propagated mid-time uncertainty.
records = detail["records"]
comp = [r for r in records if r["sourceId"] == "nasa-pscomppars"]
default = [r for r in records if r["sourceId"] == "nasa-ps" and r.get("isSourceDefault")]
def has_both(r):
    v = r.get("values") or {}
    return v.get("pl_orbper") is not None and v.get("pl_tranmid") is not None
print("PSCompPars has both:", any(has_both(r) for r in comp), "| PS default has both:", any(has_both(r) for r in default))
if any(has_both(r) for r in comp):
    expected = "nasa-pscomppars"
elif any(has_both(r) for r in default):
    expected = "nasa-ps"
else:
    expected = "a later step of the policy"
print("policy reaches:", expected, "| published row from:", row["sourceId"])
assert expected == "a later step of the policy" or expected == row["sourceId"]

In [ ]:
ref = eph["referenceBjd"]
P, sP, T0, sT0 = row["periodDays"], row["periodErrDays"], row["t0Bjd"], row["t0ErrDays"]
n = np.arange(math.ceil((ref - T0) / P), math.ceil((ref + 365.25 - T0) / P) + 1)
mid = T0 + n * P
sigma_min = np.sqrt(sT0 ** 2 + (n * sP) ** 2) * 1440
print("next transit after the release reference:", round(mid[0], 5), "BJD_TDB, sigma", round(sigma_min[0], 2), "min")
print("one year later sigma", round(sigma_min[-1], 2), "min")
published = config["published_sigma_minutes_at_reference"]
# The release row carries the inputs (period, epoch and their errors), not a propagated value;
# when a published propagated value is recorded in the config it must agree within tolerance.
assert published is None or abs(sigma_min[0] - published) <= config["tolerance_minutes"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mid - ref, sigma_min, "o-", ms=3, color="#264653")
ax.set_xlabel(f"days after the release reference BJD_TDB {ref}")
ax.set_ylabel("1-sigma mid-transit uncertainty (minutes)")
ax.set_title("HD 209458 b: propagated ephemeris uncertainty (WorldsIndex)")
fig.tight_layout(); fig.savefig("figure.png", dpi=120)
print("figure.png written")